# 01 — Architecture registry enumeration

This notebook confirms that the benchmark pipeline draws its candidate architectures from a single registry, and that the model registry and the configuration registry are in exact one-to-one correspondence. A mismatch here would mean a backbone could be selected for benchmarking with no default configuration to instantiate it (or vice versa).

Modules exercised: `models/__init__.py` (`MODEL_REGISTRY`, `CONFIG_REGISTRY`, `get_model`), consumed by `pipelines/benchmark_pipeline/pipeline.py` which builds its candidate list as `CONFIG_REGISTRY.keys()` minus `skip_models`.

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path("../..").resolve()))

import warnings
warnings.filterwarnings("ignore")

import numpy             as np
import torch
import matplotlib.pyplot as plt

SEED = 0
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.use_deterministic_algorithms(False)

DEVICE = torch.device("cpu")

plt.rcParams.update({
    "figure.dpi"     : 110,
    "font.size"      : 10,
    "axes.grid"      : True,
    "grid.alpha"     : 0.3,
    "axes.axisbelow" : True,
})

print("bootstrap ready  —  torch", torch.__version__, " device", DEVICE)


## The two registries

`MODEL_REGISTRY` maps a string key to a `nn.Module` subclass; `CONFIG_REGISTRY` maps the same key to its dataclass configuration type. The pipeline iterates `CONFIG_REGISTRY` to decide what to run, so any key present in one registry but absent from the other is a latent bug.

In [ ]:
from models import MODEL_REGISTRY, CONFIG_REGISTRY

model_keys  = set(MODEL_REGISTRY.keys())
config_keys = set(CONFIG_REGISTRY.keys())

print(f"models  registered : {len(model_keys)}")
print(f"configs registered : {len(config_keys)}")
print(f"keys only in models  : {sorted(model_keys - config_keys) or 'none'}")
print(f"keys only in configs : {sorted(config_keys - model_keys) or 'none'}")

assert model_keys == config_keys, "registries are not aligned"
print("\nregistries are aligned one-to-one")

## Default parameter count per architecture

Each architecture is instantiated once at its committed default configuration (the same defaults the size-matching stage later rescales). The bar chart gives a visual sense of how widely the default capacities vary before any matching is applied, which is precisely the disparity the pipeline corrects for.

In [ ]:
from models import get_model
from pipelines.benchmark_pipeline.sizing import _IMAGE_SIZE_MODELS

IMAGE_SIZE   = 64
IN_CHANNELS  = 9
OUT_CHANNELS = 15

param_counts = {}
for name in CONFIG_REGISTRY:
    overrides = {"in_channels": IN_CHANNELS, "out_channels": OUT_CHANNELS}
    if name in _IMAGE_SIZE_MODELS:
        overrides["image_size"] = IMAGE_SIZE

    model, _              = get_model(name, **overrides)
    param_counts[name]    = sum(p.numel() for p in model.parameters())
    del model

for name, count in sorted(param_counts.items(), key=lambda kv: kv[1]):
    print(f"{name:22s} {count:>12,}")

In [ ]:
ordered = sorted(param_counts.items(), key=lambda kv: kv[1])
names   = [n for n, _ in ordered]
values  = [v for _, v in ordered]

fig, ax = plt.subplots(figsize=(8, 6))
ax.barh(names, np.array(values) / 1e6, color="#3b6ea5")
ax.set_xlabel("default parameters (millions)")
ax.set_title(f"Default capacity per architecture  (in={IN_CHANNELS}, out={OUT_CHANNELS})")
fig.tight_layout()
plt.show()

## Expected visual outcome

The alignment check prints empty difference sets and the assertion passes, confirming the registries are exact mirrors. The horizontal bar chart shows all twenty-one architectures with default parameter counts spanning more than an order of magnitude, motivating the later capacity-matching stage.